# Measure How Stale a Quorum Read Can Be

Companion notebook for the [Quorums and Tunable Consistency](https://codemia.io/courses/distributed_systems/dst_quorums_and_tunable_consistency) lesson on Codemia.

**What you will build**: a Monte Carlo model of a leaderless store with `N` replicas. A write goes to every replica and is acknowledged after `W` of them apply it. A read asks `R` replicas and returns the newest version among their answers. You will measure the latency of each configuration, the probability that a read issued shortly after a write returns the old value, and see why `W + R > N` removes staleness while `R = W = 1` is fast and usually, but not always, fresh.

No GPU. Standard library, `numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

## 1. Replica latencies

Each replica takes a random time to apply a write and to answer a read. Real latency distributions have long tails, so use a lognormal with a median of a few milliseconds and an occasional slow replica, for example one in a garbage collection pause.

In [ ]:
def replica_ms(size, median=3.0, sigma=0.6, p_slow=0.02, slow_ms=80.0):
    base = rng.lognormal(np.log(median), sigma, size)
    slow = rng.random(size) < p_slow
    return base + slow * rng.exponential(slow_ms, size)

sample = replica_ms(100_000)
print("median %.1f ms, p99 %.1f ms, p99.9 %.1f ms" % tuple(np.percentile(sample, [50, 99, 99.9])))
plt.figure(figsize=(6, 3))
plt.hist(sample, bins=np.logspace(-0.3, 2.5, 80))
plt.xscale("log"); plt.xlabel("replica response (ms)"); plt.ylabel("count"); plt.title("One replica, long tail")
plt.show()

## 2. Quorum latency

A quorum operation waits for the `k`th fastest of `N` replies. Waiting for one of three hides a slow replica, waiting for all three exposes every pause.

In [ ]:
def quorum_latency(N, k, trials=200_000):
    lat = np.sort(replica_ms((trials, N)), axis=1)
    return lat[:, k - 1]

N = 3
for k in range(1, N + 1):
    l = quorum_latency(N, k)
    print("wait for %d of %d: median %5.1f ms, p99 %6.1f ms" % (k, N, np.median(l), np.percentile(l, 99)))

## 3. Staleness after a write

Now model the race. At time 0 a client writes version 2 over version 1. Replica `i` applies it at time `a_i`. The write is acknowledged at the `W`th smallest `a_i`. Another client starts a read `t` milliseconds after the acknowledgment. The read request reaches replica `i` after a delay `d_i`, and that replica answers with version 2 only if it has already applied the write, that is if `a_i <= ack + t + d_i`. The read takes the first `R` answers to arrive and returns the newest version among them.

This is the probabilistically bounded staleness model. It ignores read repair and hinted handoff, which only make reads fresher.

In [ ]:
def p_stale(N, R, W, t_ms, trials=200_000):
    apply_ = replica_ms((trials, N))
    ack = np.sort(apply_, axis=1)[:, W - 1:W]
    req = replica_ms((trials, N)) / 2          # one way request delay
    resp = replica_ms((trials, N)) / 2         # one way response delay
    fresh = apply_ <= ack + t_ms + req
    arrival = t_ms + req + resp
    order = np.argsort(arrival, axis=1)[:, :R]
    seen = np.take_along_axis(fresh, order, axis=1)
    return 1.0 - seen.any(axis=1).mean()

for R, W in [(1, 1), (1, 2), (2, 1), (2, 2), (1, 3), (3, 1)]:
    print("N=3 R=%d W=%d  overlap=%-5s  P(stale read right after ack) = %.4f" % (R, W, R + W > 3, p_stale(3, R, W, 0.0)))

Every configuration with `R + W > N` reports zero: some replica in the read quorum also acknowledged the write. The others are often stale at that instant: with `R = W = 1`, the acknowledging replica is usually the only one that has the write, so a read that lands elsewhere misses it nearly half the time. Many systems still accept `R = W = 1` for data that tolerates it, because the window is short. The useful question is how quickly it closes.

In [ ]:
ts = np.array([0, 0.5, 1, 2, 3, 5, 8, 12, 20, 35, 60])
plt.figure(figsize=(7, 3.6))
for R, W in [(1, 1), (1, 2), (2, 1)]:
    plt.plot(ts, [p_stale(3, R, W, t, trials=100_000) for t in ts], marker="o", label="R=%d W=%d" % (R, W))
plt.yscale("log"); plt.xlabel("time between write ack and read start (ms)"); plt.ylabel("P(read is stale)")
plt.title("Staleness closes in milliseconds, but the slow tail keeps it open")
plt.legend(); plt.grid(alpha=0.3, which="both")
plt.show()

## 4. Choosing a configuration

Put latency and staleness side by side. For each `(R, W)` with `N = 3`, report the p99 write latency, the p99 read latency and the staleness right after the write.

In [ ]:
print("R W  write p99  read p99  stale@0ms")
for R in range(1, 4):
    for W in range(1, 4):
        w99 = np.percentile(quorum_latency(3, W, 100_000), 99)
        r99 = np.percentile(quorum_latency(3, R, 100_000), 99)
        print("%d %d  %7.1f   %7.1f   %.4f" % (R, W, w99, r99, p_stale(3, R, W, 0.0, 100_000)))

## 5. Exercises

1. Raise `p_slow` to 0.1, as for a replica in a degraded zone. Which configuration's p99 suffers most, and does `R = 2, W = 2` still beat `R = 1, W = 3` on writes?
2. Move to `N = 5`. Find the cheapest `(R, W)` that is never stale and compare its p99 latencies with `N = 3`, `R = W = 2`.
3. Add **sloppy quorums**: during a partition, writes are acknowledged by stand in nodes that reads do not contact. Show that `R + W > N` stops guaranteeing fresh reads.